In [1]:
import rectanglepy as rectangle
import pandas as pd
import anndata as ad
import pickle
import scanpy as sc
import scipy.io
import mygene
import numpy as np

In [ ]:
counts = scipy.io.mmread("sampled_cattle_atlas/counts.mtx").T.tocsr()
genes = pd.read_csv("sampled_cattle_atlas/genes.csv")
meta = pd.read_csv("sampled_cattle_atlas/metadata.csv", index_col=0)

adata = ad.read_h5ad("sampled_cattle_atlas.h5ad")
#adata = sc.AnnData(X=counts, obs=meta, var=genes.set_index("gene"))
#adata.write_h5ad("sampled_cattle_atlas.h5ad")

/tmp/ipykernel_3140294/3595591599.py:3: DtypeWarning: Columns (9,12,14,16,18,20,25,37,39,41,43,45,47,49,51,53,55,57,59,61,63,65,67,70,72,74,76,78,80,82,84,86,88,90,92,94,96,98,100,102,104,106,108,110,112,113,115,117,119,121,123,125,127,129,131,133,135,137,139,141,143,145,147,149,151,153,155,157,159,161,163,165,167,169,171,173,175,177,179,181,183,185,187,189,191,193,195,197,199,201,204,206,208,210,212,214,216,218,220,222,224,226,228,230,232,234,237,239,241,243,245,247,249,251) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv("sampled_cattle_atlas/metadata.csv", index_col=0)


In [3]:
mg = mygene.MyGeneInfo()

var_names = adata.var_names.tolist()
is_ensembl = [g.startswith("ENSBTAG") for g in var_names]
symbols = [g for g, e in zip(var_names, is_ensembl) if not e]

def parse_hits(results):
    mapping = {}
    for hit in results:
        if hit.get("notfound"): continue
        ensembl = hit.get("ensembl")
        # list = multiple Ensembl IDs = asNA
        gene_id = ensembl.get("gene") if isinstance(ensembl, dict) else None
        if gene_id:
            mapping[hit["query"]] = gene_id
    return mapping

symbol_to_ensembl = parse_hits(
    mg.querymany(symbols, scopes="symbol", fields="ensembl.gene", species=9913)
)

# apply mapping
new_names = [symbol_to_ensembl.get(g, g) for g in var_names]

# keep only deduplicated ENSBTAGs (first occurrence wins, mirrors R duplicated())
seen = set()
keep = []
for i, name in enumerate(new_names):
    if name.startswith("ENSBTAG") and name not in seen:
        seen.add(name)
        keep.append(i)

n_already = sum(is_ensembl)
n_mapped = len(symbol_to_ensembl)
n_unmapped = len(symbols) - n_mapped
n_dup = sum(1 for n in new_names if n.startswith("ENSBTAG")) - len(keep)

adata = adata[:, keep]

print(f"ID mapping: {n_already}/{len(var_names)} already Ensembl IDs, "
      f"{n_mapped} gene symbols mapped uniquely to an Ensembl ID, "
      f"{n_unmapped} symbols could not be mapped and dropped. "
      f"{n_dup} ids were duplicated and dropped as well.")

285 input query terms found dup hits:	[('SMIM34', 2), ('C1H3orf70', 2), ('CLDN14', 2), ('ASMT', 2), ('TAGLN3', 2), ('LSAMP', 2), ('SRPRB',
378 input query terms found no hit:	['CSTB', 'CSTB.1', 'ECE2.1', 'COLQ.1', 'GTPBP6', 'ACAP2.1', 'BEX2', 'BEX2.1', 'IGSF1.1', 'ZNF75D.1',


ID mapping: 5469/22825 already Ensembl IDs, 16654 gene symbols mapped uniquely to an Ensembl ID, 702 symbols could not be mapped and dropped. 206 ids were duplicated and dropped as well.


In [5]:
# bulk matrices
bulk_adipo = pd.read_csv('/nfs/data2/huji_artificial_meat/adipogenesis/adipogenesis_tpm.tsv', index_col=0, sep='\t')
bulk_myo   = pd.read_csv('/nfs/data2/huji_artificial_meat/myogenesis/nfcore_out/star_salmon/salmon.merged.gene_tpm.tsv', index_col=0, sep='\t')
bulk_myo = bulk_myo.drop(columns=['gene_name'])

# metadata
meta_adipo = pd.read_csv('/nfs/data2/huji_artificial_meat/adipogenesis/sample_sheet_cleaned.csv', index_col=0)
meta_myo   = pd.read_csv('/nfs/data2/huji_artificial_meat/myogenesis/sample_sheet_cleaned.csv', index_col=0)

meta_adipo.columns = meta_adipo.columns.str.lower().str.replace(r'[^a-z0-9]', '_', regex=True).str.strip('_')
meta_myo.columns   = meta_myo.columns.str.lower().str.replace(r'[^a-z0-9]', '_', regex=True).str.strip('_')

meta_adipo = meta_adipo[['library_name', 'sample_id', 'title', 'group', 'bio_replicate', 'individual', 'treatment', 'day']]
meta_adipo['day'] = pd.Categorical(meta_adipo['day'], categories=['D0', 'D2', 'D6', 'D14'], ordered=True)
meta_adipo['short_id'] = meta_adipo['library_name'].str.replace("Sample ", "S", regex=True)

meta_myo = meta_myo[['library_name', 'sample_id', 'group', 'day', 'bio_replicate', 'treatment']]
meta_myo['day'] = pd.Categorical(meta_myo['day'], categories=['d0', 'd3', 'd7', 'd21'], ordered=True)
meta_myo['group'] = meta_myo['group'].str.replace(r'(?!Sat)D\d*', '', regex=True).str.split('_').str[-1]

In [ ]:
est_adipo, signature_result = rectangle.rectangle(adata, bulk_adipo.T)

2026-07-24 17:43:55.771 | INFO     | rectanglepy.pp.create_signature:build_rectangle_signatures:329 - Using 1037 common genes between bulks and single-cell data
2026-07-24 17:43:57.038 | INFO     | rectanglepy.pp.create_signature:_de_analysis:202 - Starting DE analysis
/nfs/home/students/t.tabanli/.conda/envs/deconv_rectangle/lib/python3.12/site-packages/rectanglepy/pp/create_signature.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  bootstrapped_signature[f"{celltype}_{i}"] = list(summed_rows)
/nfs/home/students/t.tabanli/.conda/envs/deconv_rectangle/lib/python3.12/site-packages/rectanglepy/pp/create_signature.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor pe

In [1]:
# save results as pickle
with open('rectangle_results/rectangle_signature.pkl', 'wb') as f:
    pickle.dump(signature_result, f)

# save adipo estimations as CSV
est_adipo.reset_index().to_csv('rectangle_results/rectangle_estimations_adipo.csv', index=False)

# for myo deconv:
with open('rectangle_results/rectangle_signature.pkl', 'rb') as f:
    signature_result = pickle.load(f)

est_myo, _ = rectangle.tl.deconvolution(signature_result, bulk_myo.T)
# save myo estimations as CSV
est_myo.reset_index().to_csv('rectangle_results/rectangle_estimations_myo.csv', index=False)

NameError: name 'pickle' is not defined